In [1]:
from datasets import load_dataset
import pandas as pd
import datasets
import re

datasets.utils.logging.set_verbosity_error()

/home/dhanush/CDF_RAG/MYNE/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("=" * 60)
print("Loading MedMCQA...")
print("=" * 60)

dataset = load_dataset("openlifescienceai/medmcqa")

print(dataset)

Loading MedMCQA...
DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 182822
    })
    test: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 6150
    })
    validation: Dataset({
        features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
        num_rows: 4183
    })
})


In [3]:
split = "train"

rows = []

for idx, sample in enumerate(dataset[split]):

    options = [
        sample["opa"],
        sample["opb"],
        sample["opc"],
        sample["opd"],
    ]

    answer = options[sample["cop"]]

    rows.append({
        "id": idx,

        "question": sample["question"],

        "option_a": sample["opa"],
        "option_b": sample["opb"],
        "option_c": sample["opc"],
        "option_d": sample["opd"],

        "answer": answer,

        "subject": sample.get("subject_name", ""),
        "topic": sample.get("topic_name", ""),
        "explanation": sample.get("exp", "")
    })

df = pd.DataFrame(rows)
# Replace all NaN values with empty strings
df = df.fillna("")

# Make sure all text columns are strings
text_columns = [
    "question",
    "option_a",
    "option_b",
    "option_c",
    "option_d",
    "answer",
    "subject",
    "topic",
    "explanation"
]

for col in text_columns:
    df[col] = df[col].astype(str)
    
print(df.head())
print(df.shape)

   id                                           question  \
0   0  Chronic urethral obstruction due to benign pri...   
1   1  Which vitamin is supplied from only animal sou...   
2   2  All of the following are surgical options for ...   
3   3  Following endaerectomy on the right common car...   
4   4   Growth hormone has its effect on growth through?   

                     option_a                   option_b         option_c  \
0                 Hyperplasia                  Hyperophy          Atrophy   
1                   Vitamin C                 Vitamin B7      Vitamin B12   
2  Adjustable gastric banding  Biliopancreatic diversion  Duodenal Switch   
3  Central aery of the retina          Infraorbital aery    Lacrimal aery   
4                    Directly                      IG1-1        Thyroxine   

                     option_d                      answer        subject  \
0                    Dyplasia                     Atrophy        Anatomy   
1                   Vita

In [4]:
df["document"] = (
    "Question:\n" + df["question"] +

    "\n\nCorrect Answer:\n" + df["answer"] +

    "\n\nExplanation:\n" + df["explanation"] +

    "\n\nSubject:\n" + df["subject"] +

    "\n\nTopic:\n" + df["topic"]
)

df.head()

,id,question,option_a,option_b,option_c,option_d,answer,subject,topic,explanation,document
0,0,Chronic urethral obstruction due to benign pri...,Hyperplasia,Hyperophy,Atrophy,Dyplasia,Atrophy,Anatomy,Urinary tract,Chronic urethral obstruction because of urinar...,Question:\nChronic urethral obstruction due to...
1,1,Which vitamin is supplied from only animal sou...,Vitamin C,Vitamin B7,Vitamin B12,Vitamin D,Vitamin B12,Biochemistry,Vitamins and Minerals,Ans. (c) Vitamin B12 Ref: Harrison's 19th ed. ...,Question:\nWhich vitamin is supplied from only...
2,2,All of the following are surgical options for ...,Adjustable gastric banding,Biliopancreatic diversion,Duodenal Switch,Roux en Y Duodenal By pass,Roux en Y Duodenal By pass,Surgery,Surgical Treatment Obesity,"Ans. is 'd' i.e., Roux en Y Duodenal Bypass Ba...",Question:\nAll of the following are surgical o...
3,3,Following endaerectomy on the right common car...,Central aery of the retina,Infraorbital aery,Lacrimal aery,Nasociliary aretry,Central aery of the retina,Ophthalmology,,The central aery of the retina is a branch of ...,Question:\nFollowing endaerectomy on the right...
4,4,Growth hormone has its effect on growth through?,Directly,IG1-1,Thyroxine,Intranuclear receptors,IG1-1,Physiology,,"Ans. is 'b' i.e., IGI-1GH has two major functi...",Question:\nGrowth hormone has its effect on gr...


In [5]:
OUTPUT = "processed_medmcqa.csv"

df.to_csv(
    OUTPUT,
    index=False
)

print("=" * 60)
print(df.shape)
print("Saved :", OUTPUT)
print("=" * 60)

df.head()

(182822, 11)
Saved : processed_medmcqa.csv


,id,question,option_a,option_b,option_c,option_d,answer,subject,topic,explanation,document
0,0,Chronic urethral obstruction due to benign pri...,Hyperplasia,Hyperophy,Atrophy,Dyplasia,Atrophy,Anatomy,Urinary tract,Chronic urethral obstruction because of urinar...,Question:\nChronic urethral obstruction due to...
1,1,Which vitamin is supplied from only animal sou...,Vitamin C,Vitamin B7,Vitamin B12,Vitamin D,Vitamin B12,Biochemistry,Vitamins and Minerals,Ans. (c) Vitamin B12 Ref: Harrison's 19th ed. ...,Question:\nWhich vitamin is supplied from only...
2,2,All of the following are surgical options for ...,Adjustable gastric banding,Biliopancreatic diversion,Duodenal Switch,Roux en Y Duodenal By pass,Roux en Y Duodenal By pass,Surgery,Surgical Treatment Obesity,"Ans. is 'd' i.e., Roux en Y Duodenal Bypass Ba...",Question:\nAll of the following are surgical o...
3,3,Following endaerectomy on the right common car...,Central aery of the retina,Infraorbital aery,Lacrimal aery,Nasociliary aretry,Central aery of the retina,Ophthalmology,,The central aery of the retina is a branch of ...,Question:\nFollowing endaerectomy on the right...
4,4,Growth hormone has its effect on growth through?,Directly,IG1-1,Thyroxine,Intranuclear receptors,IG1-1,Physiology,,"Ans. is 'b' i.e., IGI-1GH has two major functi...",Question:\nGrowth hormone has its effect on gr...
